
# GI Trials — ZIP Extraction, Coverage, and Enrollment (REVISED)

**What this notebook does** (Run All top-to-bottom):

1. **ZCTA setup**: loads 2020 TIGER/Line ZCTA geometries and builds a **valid US ZIP whitelist** + centroids.
2. **Trials combine & US site extraction**: loads your six CSVs, parses the **Locations** field, keeps **U.S. sites only**, validates ZIPs against ZCTAs, and deduplicates **(NCT, ZIP, cancer group)**.
3. **ZIP universe**: merges **RUCA 2020** (Urban/Rural) and **ACS 2023** (population, median household income, race/ethnicity). Also computes population-weighted income quartiles.
4. **Coverage**: computes **30/60 mile** coverage for **Any GI** and by cancer, including Urban/Rural, Income Q1/Q4, Hispanic, Black.
5. **Phase & Enrollment**: robust **phase normalization** (handles `PHASE1|PHASE2`, `EARLY_PHASE1`, roman numerals, A/B suffixes) and **enrollment totals** for Phase 1/2/3 per group and for **Any GI (unique NCT)**.
6. **QC & Validation**: checks for non-US leakage, invalid ZIPs, duplicates, geocoding completeness, monotonic coverage, and runs **sensitivity/uncertainty** tests.

**Key fixes vs prior versions**
- Tokenizes `Phases` like `PHASE1|PHASE2` and maps **Early Phase 1 → Phase 1**; **1/2 → 2**, **2/3 → 3**.
- Enforces **US-only site tokens** and validates **5-digit ZIPs** against ZCTA whitelist.
- Avoids pandas `FutureWarning` by using aggregation without `GroupBy.apply` on grouping columns.
- Writes clean outputs to the `out/` folder and provides light-weight QC artifacts for audit.

> ⚠️ Assumes your folder structure:
>
> - `data/` containing the six CSVs: `Colorectal.csv`, `Pancreatic Cancer.csv`, `Gastric Cancer.csv`, `Esophageal cancer.csv`, `Hepatocellular.csv`, `cholangiocarcinoma.csv`
> - `data/RUCA-codes-2020-zipcode.csv`
> - `tl_2020_us_zcta510/` (or `data/tl_2020_us_zcta510/`) with the ZCTA shapefile bundle.


In [71]:

# ==============================
# Config
# ==============================
import os, re, warnings, glob
import numpy as np
import pandas as pd

# DATA_DIR = "../data"
DATA_DIR = "../asco-data"
FILES = [
    "car-t_final.csv"
]

# FILES = [
#     "Anal cancer_clean.csv",
#     "Appendiceal cancer.csv",
#     "Biliary Duct cancer_clean.csv",
#     "Colorectal cancer_clean.csv",
#     "Esophageal_Clean.csv",
#     "Gastric Cancer_clean.csv",
#     "Gastroesophageal_clean.csv",
#     "Hepatocellular cancer_clean.csv",
#     "Pancreatic_clean.csv",
#     "Small bowel adenocarcinoma_Clean.csv"
# ]

def _norm(s): 
    import re
    return re.sub(r"[^a-z0-9]+","_", str(s).lower()).strip("_")

# F2KEY = {
#     _norm("Colorectal"):               "colorectal_cancer",
#     _norm("Pancreatic Cancer"):        "pancreatic_cancer",
#     _norm("Gastric Cancer"):           "gastric_cancer",
#     _norm("Esophageal cancer"):        "oesophageal_cancer",
#     _norm("Hepatocellular"):           "hepatocellular_cancer",
#     _norm("cholangiocarcinoma"):       "cholangiocarcinoma",
# }

# F2KEY = {
#     _norm("Anal cancer_clean"):               "anal_cancer",
#     _norm("Appendiceal cancer"):              "appendiceal_cancer",
#     _norm("Biliary Duct cancer_clean"):       "biliary_duct_cancer",
#     _norm("Colorectal cancer_clean"):         "colorectal_cancer",
#     _norm("Esophageal_Clean"):                "esophageal_cancer",
#     _norm("Gastric Cancer_clean"):            "gastric_cancer",
#     _norm("Gastroesophageal_clean"):          "gastroesophageal_cancer",
#     _norm("Hepatocellular cancer_clean"):     "hepatocellular_cancer",
#     _norm("Pancreatic_clean"):                "pancreatic_cancer",
#     _norm("Small bowel adenocarcinoma_Clean"): "adenocarcinoma",


# }


F2KEY = {
    _norm("car _t final study"):               "Car_T",
}
# LABELS = {
#     "anal_cancer":"Anal_cancer",
#     "appendiceal_cancer":"Appendiceal_cancer",
#     "biliary_duct_cancer":"BiliaryDuct_cancer",
#     "colorectal_cancer":"Colorectal_cancer",
#     "esophageal_cancer":"Esophageal_cancer",
#     "gastric_cancer":"Gastric_cancer",
#     "gastroesophageal_cancer":"Gastroesophageal_cancer",
#     "hepatocellular_cancer":"Hepatocellular_cancer",
#     "pancreatic_cancer":"Pancreatic_cancer",
#     "adenocarcinoma":"Adenocarcinoma"
    
    

# }

LABELS = {
    "Car_T":"car_t"
}

os.makedirs("out", exist_ok=True)


In [32]:
# --- ADD TO HELPERS SECTION ---
# --- ADD TO HELPERS / CONFIG SECTION ---
COMMON_CANCERS = {
    "colorectal_cancer", "pancreatic_cancer", "hepatocellular_cancer", 
    "gastric_cancer", "esophageal_cancer", "gastroesophageal_cancer"
}

RARE_CANCERS = {
    "anal_cancer", "appendiceal_cancer", "biliary_duct_cancer", "adenocarcinoma"
}

def get_rarity_group(ctype):
    if ctype in COMMON_CANCERS: return "Common"
    if ctype in RARE_CANCERS: return "Rare"
    return "Other"

def map_funder_type(funder_str):
    if not isinstance(funder_str, str): return "Other"
    f = funder_str.upper()
    if "NIH" in f: return "NIH"
    if "INDUSTRY" in f: return "Industry"
    if any(x in f for x in ["FED", "U.S. FED", "FDA", "VA", "DOD"]): return "Fed"
    return "Other"

In [72]:

# ==============================
# PREREQ: ZCTA centroids + valid ZIP whitelist
# ==============================
import geopandas as gpd

CANDIDATE_DIRS = [
    "../data/tl_2020_us_zcta510"
]
ZCTA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(d)), None)
assert ZCTA_DIR is not None, f"ZCTA shapefile dir not found in: {CANDIDATE_DIRS}"

zcta = gpd.read_file(ZCTA_DIR).to_crs(epsg=4326)
id_col = "ZCTA5CE10" if "ZCTA5CE10" in zcta.columns else (
         "ZCTA5CE20" if "ZCTA5CE20" in zcta.columns else None)
assert id_col is not None, f"Could not find ZCTA id column in {list(zcta.columns)[:10]}"

zcta["centroid"] = zcta.geometry.centroid
zcta["lon"] = zcta["centroid"].x
zcta["lat"] = zcta["centroid"].y

zip_centroids = zcta[[id_col, "lat", "lon"]].rename(columns={id_col:"zip"}).copy()
zip_centroids["zip"] = zip_centroids["zip"].astype(str).str.zfill(5)

valid_us_zips = set(zip_centroids["zip"].tolist())
print("zip_centroids:", zip_centroids.shape, "| valid_us_zips:", len(valid_us_zips))


/var/folders/4l/64tth9h948n3dbq1py15y2qr0000gn/T/ipykernel_2282/3573664915.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zcta["centroid"] = zcta.geometry.centroid


zip_centroids: (33144, 3) | valid_us_zips: 33144


In [73]:

# ==============================
# Helpers: US site detection + ZIP extraction
# ==============================
import re

US_COUNTRY_PAT = re.compile(r"\b(united\s*states|u\.s\.a\.?|u\.s\.)\b", re.I)
US_STATE_ABBRS = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC",
    "ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"
}
US_STATE_NAMES = {
    "alabama","alaska","arizona","arkansas","california","colorado","connecticut","delaware",
    "district of columbia","florida","georgia","hawaii","idaho","illinois","indiana","iowa",
    "kansas","kentucky","louisiana","maine","maryland","massachusetts","michigan","minnesota",
    "mississippi","missouri","montana","nebraska","nevada","new hampshire","new jersey",
    "new mexico","new york","north carolina","north dakota","ohio","oklahoma","oregon",
    "pennsylvania","rhode island","south carolina","south dakota","tennessee","texas","utah",
    "vermont","virginia","washington","west virginia","wisconsin","wyoming"
}
ZIP5_RE = re.compile(r"(?<!\d)(\d{5})(?!\d)")

def is_us_site(s: str) -> bool:
    if not isinstance(s, str): return False
    low = s.lower()
    if US_COUNTRY_PAT.search(low): return True
    m = re.findall(r",\s*([A-Z]{2})\b", s)
    if any(x in US_STATE_ABBRS for x in m): return True
    if any(name in low for name in US_STATE_NAMES): return True
    return False

def tokenize_locations_field(s: str):
    if not isinstance(s, str): return []
    return re.split(r"\s*\|\s*", s.strip())

def extract_zip(s: str):
    if not isinstance(s, str): return None
    m = ZIP5_RE.findall(s)
    return m[0] if m else None


In [74]:

# ==============================
# Combine CSVs + US-only site ZIPs + *_with_zip.csv
# ==============================
frames = []
sites_rows = []
out_dir = os.path.join(DATA_DIR, "output/asco")
os.makedirs(out_dir, exist_ok=True)

for f in FILES:
    print(f)
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        warnings.warn(f"Missing file: {p}")
        continue
    df = pd.read_csv(p, dtype=str)


    # tolerant column mapping
    nct_col = next((c for c in df.columns if c.lower().startswith("nct")), None)
    phases_col = next((c for c in df.columns if c.lower()=="phases"), None)
    loc_col = "Locations" if "Locations" in df.columns else ("Location" if "Location" in df.columns else None)
    if nct_col is None or loc_col is None:
        raise ValueError(f"{f}: need NCT Number and Locations/Location columns")

    key = F2KEY.get(_norm(os.path.splitext(f)[0]), "other_gi")

    # master for later phase/enrollment work
    tmp_master = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "cancer_type": key,
        "phase_raw": df[phases_col].astype(str) if phases_col else np.nan
    })
    frames.append(tmp_master[["nct_id","cancer_type","phase_raw"]])

    # explode locations → tokens
    tok = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "locations": df[loc_col].astype(str)
    })
    tok["loc_list"] = tok["locations"].apply(tokenize_locations_field)
    tok = tok.explode("loc_list", ignore_index=True).dropna(subset=["loc_list"])

    # US-only + ZIP extraction + ZCTA whitelist
    tok = tok[tok["loc_list"].apply(is_us_site)]
    tok["zip"] = tok["loc_list"].apply(extract_zip).astype(str).str.zfill(5)
    tok = tok[tok["zip"].isin(valid_us_zips)]

    # save *_with_zip.csv preserving original columns and adding Zipcode
    out_path = os.path.join(out_dir, f"{os.path.splitext(os.path.basename(f))[0]}_with_zip.csv")
    df_out = df.copy()
    # create exploded rows aligned to tokens/zip
    exploded = tok[["nct_id","zip"]].dropna()
    exploded = exploded.drop_duplicates()
    # join back to original on NCT to duplicate rows per ZIP
    df_out = df_out.merge(exploded, left_on=nct_col, right_on="nct_id", how="inner").drop(columns=["nct_id"])
    df_out = df_out.rename(columns={"zip":"Zipcode"})
    df_out.to_csv(out_path, index=False)

    # rows for site_df
    tok["cancer_type"] = key
    tok = tok.drop_duplicates(subset=["nct_id","zip","cancer_type"])
    sites_rows.append(tok[["nct_id","cancer_type","zip"]])

# master + site_df + trial_sites
master = (pd.concat(frames, ignore_index=True)
          if frames else pd.DataFrame(columns=["nct_id","cancer_type","phase_raw"]))
master = master.sort_values(["nct_id","cancer_type"]).drop_duplicates(["nct_id","cancer_type"], keep="first")

site_df = (pd.concat(sites_rows, ignore_index=True)
           if sites_rows else pd.DataFrame(columns=["nct_id","cancer_type","zip"]))
site_df["zip"] = site_df["zip"].astype(str).str.zfill(5)

trial_sites = (site_df.drop_duplicates(["zip","cancer_type"])
               .merge(zip_centroids, on="zip", how="left")
               .dropna(subset=["lat","lon"]))

print("Master trials:", master.shape, "| Site rows (US ZIP validated):", site_df.shape, "| Unique geocoded sites:", len(trial_sites))


car-t_final.csv
Master trials: (189, 3) | Site rows (US ZIP validated): (704, 3) | Unique geocoded sites: 169


In [75]:

# ==============================
# Build ZIP/ZCTA universe (RUCA + ACS) -> zip_universe
# ==============================
import requests

ruca_path = os.path.join("../data", "RUCA-codes-2020-zipcode.csv")
ruca_raw = pd.read_csv(ruca_path, dtype=str)
def _normc(s): 
    import re; 
    return re.sub(r"[^a-z0-9]+","_", str(s).lower()).strip("_")
ruca = ruca_raw.rename(columns={c: _normc(c) for c in ruca_raw.columns})

# pick RUCA and ZIP columns heuristically
cand_ruca_cols = [c for c in ruca.columns if ("primary" in c and "ruca" in c)]               or [c for c in ruca.columns if c.endswith("ruca") or c.startswith("ruca")]
assert cand_ruca_cols, f"Could not find a RUCA code column in {list(ruca.columns)[:10]}"
ruca_code_col = cand_ruca_cols[0]

zip_col = None
for c in ruca.columns:
    m = ruca[c].astype(str).str.extract(r"(\d{5})")[0]
    if m.notna().mean() >= 0.9:
        zip_col = c; break
assert zip_col, "Could not find a ZIP column in RUCA CSV."

def _ruca_to_urb(x: str) -> str:
    try:
        v = float(str(x).strip())
    except Exception:
        return "Unknown"
    if 1.0 <= v <= 3.0:  return "Urban"
    if 4.0 <= v <= 10.0: return "Rural"
    return "Unknown"

ruca_df = pd.DataFrame({
    "zip": ruca[zip_col].astype(str).str.extract(r"(\d{5})")[0].str.zfill(5),
    "PrimaryRUCA": ruca[ruca_code_col].astype(str)
}).dropna(subset=["zip"]).drop_duplicates(subset=["zip"])
ruca_df["rural_urban"] = ruca_df["PrimaryRUCA"].apply(_ruca_to_urb)

# ACS fetch (2023); optional API key via env
YEAR = 2023
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "").strip()

def fetch_acs_vars(year: int, vars_list: list[str]) -> pd.DataFrame:
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": ",".join(["NAME"] + vars_list), "for": "zip code tabulation area:*"}
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    r = requests.get(base, params=params, timeout=60)
    r.raise_for_status()
    rows = r.json()
    cols = rows[0]
    data = rows[1:]
    df = pd.DataFrame(data, columns=cols).rename(columns={"zip code tabulation area":"zip"})
    df["zip"] = df["zip"].astype(str).str.zfill(5)
    for v in vars_list:
        df[v] = pd.to_numeric(df[v], errors="coerce")
    return df[["zip"] + vars_list]

pop_df = fetch_acs_vars(YEAR, ["B01003_001E"]).rename(columns={"B01003_001E":"population"})
inc    = fetch_acs_vars(YEAR, ["B19013_001E"]).rename(columns={"B19013_001E":"mhi"})
eth    = fetch_acs_vars(YEAR, ["B03003_001E","B03003_003E"]).rename(
            columns={"B03003_001E":"pop_total_eth","B03003_003E":"pop_hispanic"})
blk    = fetch_acs_vars(YEAR, ["B02001_001E","B02001_003E"]).rename(
            columns={"B02001_001E":"pop_total_race","B02001_003E":"pop_black"})

zip_universe = (
    zip_centroids
      .merge(ruca_df[["zip","rural_urban"]], on="zip", how="left")
      .merge(pop_df, on="zip", how="left")
      .merge(inc,    on="zip", how="left")
      .merge(eth,    on="zip", how="left")
      .merge(blk,    on="zip", how="left")
)

for c in ["population","mhi","pop_total_eth","pop_hispanic","pop_total_race","pop_black"]:
    if c in zip_universe.columns:
        zip_universe[c] = pd.to_numeric(zip_universe[c], errors="coerce")

print("ZIP universe rows:", len(zip_universe),
      "| pop present:", f"{100*zip_universe['population'].notna().mean():.1f}%",
      "| mhi present:", f"{100*zip_universe['mhi'].notna().mean():.1f}%")

# Population-weighted income quartiles
def weighted_quantile(values, weights, qs):
    v = np.asarray(values); w = np.asarray(weights)
    idx = np.argsort(v); v=v[idx]; w=w[idx]
    cw = np.cumsum(w) - 0.5*w
    cw = cw / np.sum(w)
    return np.interp(qs, cw, v)

mask = zip_universe["mhi"].notna() & zip_universe["population"].notna()
if mask.any():
    q1,q2,q3 = weighted_quantile(zip_universe.loc[mask,"mhi"],
                                 zip_universe.loc[mask,"population"], [0.25,0.5,0.75])
    zip_universe["inc_quartile"] = pd.cut(zip_universe["mhi"],
                                          bins=[-np.inf,q1,q2,q3,np.inf],
                                          labels=["Q1 (Lowest)","Q2","Q3","Q4 (Highest)"])
else:
    zip_universe["inc_quartile"] = np.nan


ZIP universe rows: 33144 | pop present: 99.3% | mhi present: 99.3%


In [76]:

# ==============================
# Coverage @ 30 & 60 miles + stratified tables
# ==============================
from sklearn.neighbors import BallTree

for need in ["master","site_df","trial_sites","zip_universe"]:
    if need not in globals():
        raise RuntimeError(f"Missing `{need}`")

zip_universe["population"] = pd.to_numeric(zip_universe["population"], errors="coerce")

EARTH_RADIUS_MI = 3958.7613
RADII_MI = [30, 60]

def build_tree(df):
    pts = np.deg2rad(df[["lat","lon"]].to_numpy())
    return BallTree(pts, metric="haversine")

trees = {}
if len(trial_sites):
    for ctype, g in trial_sites.groupby("cancer_type"):
        trees[ctype] = build_tree(g)
    tree_any = build_tree(trial_sites)
else:
    tree_any = None

qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())

for r in RADII_MI:
    r_rad = r / EARTH_RADIUS_MI
    zip_universe[f"covered_any_gi_{r}mi"] = False if tree_any is None else         np.array([len(ix)>0 for ix in tree_any.query_radius(qpts, r=r_rad)], dtype=bool)
    for ctype in sorted(master["cancer_type"].unique()):
        t = trees.get(ctype)
        col = f"covered_{ctype}_{r}mi"
        zip_universe[col] = False if t is None else             np.array([len(ix)>0 for ix in t.query_radius(qpts, r=r_rad)], dtype=bool)

def pop_share(df, covered_col, weight_col="population"):
    if covered_col not in df or weight_col not in df or len(df)==0: return float("nan")
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den<=0: return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def subpop_share(df, covered_col, subpop_col):
    if covered_col not in df or subpop_col not in df: return float("nan")
    w = pd.to_numeric(df[subpop_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den<=0: return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def table_for_radius(r):
    rows = []
    keys = ["any_gi"] + list(master["cancer_type"].unique())
    for key in keys:
        label = LABELS.get(key, key)
        col = f"covered_any_gi_{r}mi" if key=="any_gi" else f"covered_{key}_{r}mi"
        pop_all  = pop_share(zip_universe, col)
        pop_urb  = pop_share(zip_universe.loc[zip_universe["rural_urban"]=="Urban"], col)
        pop_rur  = pop_share(zip_universe.loc[zip_universe["rural_urban"]=="Rural"], col)
        q1       = pop_share(zip_universe.loc[zip_universe["inc_quartile"]=="Q1 (Lowest)"], col)
        q4       = pop_share(zip_universe.loc[zip_universe["inc_quartile"]=="Q4 (Highest)"], col)
        hisp     = subpop_share(zip_universe, col, "pop_hispanic")
        black    = subpop_share(zip_universe, col, "pop_black")
        rows.append({
            "Cancer Type": label,
            "% Pop Covered": 100*pop_all if pd.notna(pop_all) else np.nan,
            "% Urban Covered": 100*pop_urb if pd.notna(pop_urb) else np.nan,
            "% Rural Covered": 100*pop_rur if pd.notna(pop_rur) else np.nan,
            "% Q1 Income Covered": 100*q1 if pd.notna(q1) else np.nan,
            "% Q4 Income Covered": 100*q4 if pd.notna(q4) else np.nan,
            "% Hispanic Covered": 100*hisp if pd.notna(hisp) else np.nan,
            "% Black Covered": 100*black if pd.notna(black) else np.nan,
        })
    return pd.DataFrame(rows)

table_30 = table_for_radius(30)
table_60 = table_for_radius(60)

table_30.to_csv(os.path.join("out", "coverage_30mi.csv"), index=False)
table_60.to_csv(os.path.join("out", "coverage_60mi.csv"), index=False)
table_60.head(10)


,Cancer Type,% Pop Covered,% Urban Covered,% Rural Covered,% Q1 Income Covered,% Q4 Income Covered,% Hispanic Covered,% Black Covered
0,any_gi,72.854692,80.584027,34.379267,54.177945,93.219386,74.149888,79.923961
1,other_gi,72.854692,80.584027,34.379267,54.177945,93.219386,74.149888,79.923961


In [65]:

# ==============================
# Phase 1/2/3: Trials & Enrollment (per group + Any GI single row)
# ==============================
# Robust parser: tokenizes 'PHASE1|PHASE2', handles EARLY_PHASE1, roman numerals, A/B suffix
import re

ROMAN = {"I":1,"II":2,"III":3,"IV":4}

def _tokenize_phases(s: str):
    if not isinstance(s, str) or not s.strip():
        return []
    t = s.upper().strip()
    t = re.sub(r'[\s,;/&]+', '|', t)
    t = t.replace('||','|').strip('|')
    return [tok for tok in t.split('|') if tok]

def _phase_int_from_token(tok: str):
    if "EARLY" in tok and "PHASE1" in tok:
        return 1
    if "PHASE0" in tok:
        return 1
    m = re.search(r'PHASE[_\s-]*([0-4]|I{1,3}|IV)\s*[AB]?$', tok)
    if not m:
        m = re.match(r'^([0-4]|I{1,3}|IV)$', tok)
    if not m:
        return None
    g = m.group(1)
    v = int(g) if g.isdigit() else ROMAN.get(g, None)
    return v if v in {1,2,3,4} else None

def normalize_phase(phases_text: str):
    toks = _tokenize_phases(phases_text)
    vals = [v for v in (_phase_int_from_token(t) for t in toks) if v is not None]
    if not vals:
        return None
    st = set(vals)
    if st == {1,2}: v = 2
    elif st == {2,3}: v = 3
    else: v = max(vals)
    return f"Phase {v}"

# Build enroll_df from source CSVs (pulls Enrollment if present)
def parse_enrollment(x):
    if pd.isna(x): return np.nan
    s = str(x)
    m = re.search(r"(\d[\d,]*)", s)
    if not m: return np.nan
    return pd.to_numeric(m.group(1).replace(",",""), errors="coerce")

rows = []
for f in FILES:
    p = os.path.join(DATA_DIR, f)
    if not os.path.exists(p):
        continue
    df = pd.read_csv(p, dtype=str)
    nct_col    = next((c for c in df.columns if c.lower().startswith("nct")), None)
    phases_col = next((c for c in df.columns if c.lower()=="phases"), None)
    enroll_col = next((c for c in df.columns if c.lower()=="enrollment"), None)
    key = F2KEY.get(_norm(os.path.splitext(f)[0]), "other_gi")
# --- MODIFY INSIDE THE LOOP IN CELL 13 ---
# Inside your loop:
# Specifically look for 'Funder Type' first as it is the most accurate
    funder_col = next((c for c in df.columns if c.lower() == "funder type"), None)
    if not funder_col:
        # Fallback to general sponsor columns if 'Funder Type' isn't there
        funder_col = next((c for c in df.columns if "funder" in c.lower() or "sponsor" in c.lower()), None)
    
    tmp = pd.DataFrame({
        "nct_id": df[nct_col].astype(str).str.strip(),
        "cancer_type": key,
        "rarity": get_rarity_group(key),
        "phase_norm": df[phases_col].apply(normalize_phase) if phases_col else np.nan,
        "enrollment_num": df[enroll_col].apply(parse_enrollment) if enroll_col else np.nan,
        "funder_type": df[funder_col].apply(map_funder_type) if funder_col else "Other"
    })
    rows.append(tmp)

enroll_df = (pd.concat(rows, ignore_index=True)
             if rows else pd.DataFrame(columns=["nct_id","cancer_type","phase_norm","enrollment_num"]))
enroll_df = enroll_df.sort_values(["nct_id","cancer_type"]).drop_duplicates(["nct_id","cancer_type"], keep="first")

phase_order = ["Phase 1","Phase 2","Phase 3"]

counts = (enroll_df.dropna(subset=["phase_norm"])
          .groupby(["cancer_type","phase_norm"])["nct_id"].nunique()
          .unstack(fill_value=0)
          .reindex(columns=phase_order, fill_value=0))
counts.columns = [f"Trials {ph}" for ph in counts.columns]

enr = (enroll_df.groupby(["cancer_type","phase_norm"])["enrollment_num"]
       .sum(min_count=1)
       .unstack()
       .reindex(columns=phase_order))
enr.columns = [f"Enrollment {ph}" for ph in enr.columns]

per_group = pd.concat([counts, enr], axis=1).reset_index()
per_group.insert(0, "Cancer Type", per_group["cancer_type"].map(LABELS).fillna(per_group["cancer_type"]))
per_group = per_group.drop(columns=["cancer_type"])

def phase_to_int(ph):
    return {"Phase 1":1, "Phase 2":2, "Phase 3":3, "Phase 4":4}.get(ph, np.nan)

gi_unique = (enroll_df
             .assign(phase_int=lambda d: d["phase_norm"].map(phase_to_int))
             .groupby("nct_id", as_index=False)
             .agg(phase_int=("phase_int","max"), enrollment_num=("enrollment_num","max")))
gi_unique["phase_norm"] = gi_unique["phase_int"].map({1:"Phase 1",2:"Phase 2",3:"Phase 3",4:"Phase 4"})

any_counts = gi_unique.groupby("phase_norm")["nct_id"].nunique()
any_enr    = gi_unique.groupby("phase_norm")["enrollment_num"].sum(min_count=1)

row = {"Cancer Type": "Any GI (Unique NCT)"}
for ph in phase_order:
    row[f"Trials {ph}"]     = int(any_counts.get(ph, 0))
    val = any_enr.get(ph, np.nan)
    row[f"Enrollment {ph}"] = float(val) if pd.notna(val) else np.nan
any_gi_one_row = pd.DataFrame([row])

per_group.to_csv(os.path.join("out","phase_enrollment_by_group.csv"), index=False)
any_gi_one_row.to_csv(os.path.join("out","phase_enrollment_any_gi_unique_row.csv"), index=False)
per_group.head(10), any_gi_one_row


(  Cancer Type  Trials Phase 1  Trials Phase 2  Trials Phase 3  \
 0       car_t             138              47               4   
 
    Enrollment Phase 1  Enrollment Phase 2  Enrollment Phase 3  
 0                5716                4912                1398  ,
            Cancer Type  Trials Phase 1  Enrollment Phase 1  Trials Phase 2  \
 0  Any GI (Unique NCT)             138              5716.0              47   
 
    Enrollment Phase 2  Trials Phase 3  Enrollment Phase 3  
 0              4912.0               4              1398.0  )

In [39]:

# ==============================
# QC battery (slim): invalid ZIPs, duplicates, geocoding, monotonic coverage
# ==============================
print("[QC] site_df rows:", len(site_df), "| unique (NCT,ZIP,group):", len(site_df.drop_duplicates(['nct_id','zip','cancer_type'])))
bad_len = (site_df["zip"].str.len()!=5).sum()
not_in_whitelist = (~site_df["zip"].isin(valid_us_zips)).sum()
dups = site_df.duplicated(subset=["nct_id","zip","cancer_type"]).sum()
print("  zip length !=5:", bad_len, "| zip not in whitelist:", not_in_whitelist, "| duplicates:", dups)

# Geocoding completeness
miss_geo = trial_sites["lat"].isna().sum() + trial_sites["lon"].isna().sum()
print("[QC] trial_sites geocoding missing (sum):", miss_geo)

# Monotonicity 30->60
cov30 = [c for c in zip_universe.columns if c.endswith("_30mi")]
cov60 = [c for c in zip_universe.columns if c.endswith("_60mi")]
if cov30 and cov60:
    probs = []
    for c30 in cov30:
        c60 = c30.replace("_30mi","_60mi")
        if c60 in zip_universe:
            bad = (zip_universe[c30] & ~zip_universe[c60]).sum()
            if bad: probs.append((c30, int(bad)))
    if probs:
        print("[QC] PROBLEM: 30mi covered but not at 60mi:", probs)
    else:
        print("[QC] OK: 60mi is a superset of 30mi for all flags")
else:
    print("[QC] Skipped monotonicity (flags not present)")


[QC] site_df rows: 4776 | unique (NCT,ZIP,group): 4776
  zip length !=5: 0 | zip not in whitelist: 0 | duplicates: 0
[QC] trial_sites geocoding missing (sum): 0
[QC] OK: 60mi is a superset of 30mi for all flags


In [40]:
# Replace the subgroup share helper with the correct logic:
def subgroup_coverage(df, covered_col: str, subgroup_col: str) -> float:
    """
    Share of a subgroup (e.g., Hispanic or Black) that lives in covered ZIPs.
    Numerator and denominator both use the subgroup column.
    """
    if any(col not in df.columns for col in [covered_col, subgroup_col]) or df.empty:
        return float("nan")
    w = pd.to_numeric(df[subgroup_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den <= 0:
        return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)


In [66]:
# ==============================
# Page-2 style coverage table (like the screenshot)
# Prereqs: master, trial_sites (with lat/lon), zip_universe (with lat/lon, population, mhi, pop_hispanic, pop_black, rural_urban, inc_quartile)
# ==============================
import numpy as np
import pandas as pd

# ---- Params
RADIUS_MI = 30   # <- set 30 or 60
INCLUDE_OTHER_GI = False

# ---- Label map (adjust if you added/removed groups)
CANCER_LABELS = LABELS
if INCLUDE_OTHER_GI:
    CANCER_LABELS["other_gi"] = "Other GI"

# ---- Safety checks
for need in ["master","trial_sites","zip_universe"]:
    if need not in globals():
        raise RuntimeError(f"Missing `{need}`. Run the combine/ZIP-universe steps first.")

# Ensure numerics present
zip_universe["population"]   = pd.to_numeric(zip_universe["population"], errors="coerce")
zip_universe["pop_hispanic"] = pd.to_numeric(zip_universe.get("pop_hispanic", np.nan), errors="coerce")
zip_universe["pop_total_eth"]= pd.to_numeric(zip_universe.get("pop_total_eth", np.nan), errors="coerce")
zip_universe["pop_black"]    = pd.to_numeric(zip_universe.get("pop_black", np.nan), errors="coerce")
zip_universe["pop_total_race"]=pd.to_numeric(zip_universe.get("pop_total_race", np.nan), errors="coerce")

# ---- If covered_* columns for this radius are missing, compute them
def ensure_coverage_flags(radius_mi: int):
    col_any = f"covered_any_gi_{radius_mi}mi"
    cols_needed = [col_any] + [f"covered_{k}_{radius_mi}mi" for k in master["cancer_type"].unique()]
    missing = [c for c in cols_needed if c not in zip_universe.columns]
    if not missing:
        return
    from sklearn.neighbors import BallTree
    EARTH_RADIUS_MI = 3958.7613
    r_rad = radius_mi / EARTH_RADIUS_MI
    qpts = np.deg2rad(zip_universe[["lat","lon"]].to_numpy())

    def tree_from(df):
        if df.empty: 
            return None
        pts = np.deg2rad(df[["lat","lon"]].to_numpy())
        return BallTree(pts, metric="haversine")

    # Any GI
    t_any = tree_from(trial_sites)
    zip_universe[col_any] = False if t_any is None else np.array([len(ix)>0 for ix in t_any.query_radius(qpts, r=r_rad)], bool)

    # Per cancer
    for key, g in trial_sites.groupby("cancer_type"):
        t = tree_from(g)
        col = f"covered_{key}_{radius_mi}mi"
        zip_universe[col] = False if t is None else np.array([len(ix)>0 for ix in t.query_radius(qpts, r=r_rad)], bool)

ensure_coverage_flags(RADIUS_MI)

# ---- Helpers
def pop_share(df: pd.DataFrame, covered_col: str, weight_col: str = "population") -> float:
    if covered_col not in df.columns or weight_col not in df.columns or df.empty:
        return float("nan")
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den <= 0:
        return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def covered_population_millions(df: pd.DataFrame, covered_col: str) -> float:
    if covered_col not in df.columns or "population" not in df.columns:
        return float("nan")
    w = pd.to_numeric(df["population"], errors="coerce").fillna(0.0)
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / 1_000_000.0)

def race_eth_share(df: pd.DataFrame, covered_col: str, num_col: str, den_col: str) -> float:
    for col in (covered_col, num_col, den_col):
        if col not in df.columns:
            return float("nan")
    num = pd.to_numeric(df[num_col], errors="coerce").fillna(0.0)
    den = pd.to_numeric(df[den_col], errors="coerce").fillna(0.0).sum()
    if den <= 0:
        return float("nan")
    c = df[covered_col].astype(bool)
    return float((num[c]).sum() / den)

def counts_for(key: str) -> tuple[int, int]:
    if key == "any_gi":
        n_trials = int(master["nct_id"].nunique())
        n_sites  = int(trial_sites["zip"].nunique())
        return n_trials, n_sites
    tf = master[master["cancer_type"] == key]
    ts = trial_sites[trial_sites["cancer_type"] == key]
    return int(tf["nct_id"].nunique()), int(ts["zip"].nunique())

# ---- Build rows
# --- UPDATED ORDERING LOGIC ---
rows = []

# 1. Define the preferred order (Any GI first, then Common, then Rare)
# We use the sets we defined earlier
priority_order = ["any_gi"] + sorted(list(COMMON_CANCERS)) + sorted(list(RARE_CANCERS))

# 2. Get what actually exists in the data
exists = set(master["cancer_type"].unique())

# 3. Build the final list: 
# First, items in priority_order that exist; 
# Then, anything else in the data we might have missed.
ordered_keys = [k for k in priority_order if k in exists or k == "any_gi"]
ordered_keys += [k for k in exists if k not in ordered_keys]

for key in ordered_keys:
    label = CANCER_LABELS.get(key, key)
    covered_col = f"covered_any_gi_{RADIUS_MI}mi" if key=="any_gi" else f"covered_{key}_{RADIUS_MI}mi"
    n_trials, n_sites = counts_for(key)

    # overall + subgroups
    pop_all  = pop_share(zip_universe, covered_col, "population")
    coveredM = covered_population_millions(zip_universe, covered_col)

    if "rural_urban" in zip_universe.columns:
        pop_urb = pop_share(zip_universe[zip_universe["rural_urban"]=="Urban"], covered_col)
        pop_rur = pop_share(zip_universe[zip_universe["rural_urban"]=="Rural"], covered_col)
    else:
        pop_urb = pop_rur = float("nan")

    if "inc_quartile" in zip_universe.columns:
        q1_df = zip_universe[zip_universe["inc_quartile"]=="Q1 (Lowest)"]
        q4_df = zip_universe[zip_universe["inc_quartile"]=="Q4 (Highest)"]
        pop_q1 = pop_share(q1_df, covered_col)
        pop_q4 = pop_share(q4_df, covered_col)
    else:
        pop_q1 = pop_q4 = float("nan")

# Old (incorrect):
# hisp = race_eth_share(zip_universe, covered_col, "pop_hispanic", "pop_total_eth")
# blk  = race_eth_share(zip_universe, covered_col, "pop_black", "pop_total_race")

# New (correct):
    hisp = subgroup_coverage(zip_universe, covered_col, "pop_hispanic")
    blk  = subgroup_coverage(zip_universe, covered_col, "pop_black")


    rows.append({
        "Cancer Type": label,
        "No. Trials (NCT IDs)": n_trials if n_trials>0 else np.nan,
        "No. Sites (ZIPs)": n_sites if n_sites>0 else np.nan,
        "% Population Covered": 100*pop_all if pd.notna(pop_all) else np.nan,
        "Covered Population (M)": coveredM if pd.notna(coveredM) else np.nan,
        "% Patients Covered (SEER Incidence)": np.nan,  # placeholder
        "% Urban Pop Covered": 100*pop_urb if pd.notna(pop_urb) else np.nan,
        "% Rural Pop Covered": 100*pop_rur if pd.notna(pop_rur) else np.nan,
        "% Lowest Income Quartile Covered": 100*pop_q1 if pd.notna(pop_q1) else np.nan,
        "% Highest Income Quartile Covered": 100*pop_q4 if pd.notna(pop_q4) else np.nan,
        "% Hispanic Pop Covered": 100*hisp if pd.notna(hisp) else np.nan,
        "% Black Pop Covered": 100*blk if pd.notna(blk) else np.nan,
    })

table = pd.DataFrame(rows)

# ---- Nicely formatted version (matches the screenshot style)
fmt = table.copy()
pct_cols = [
    "% Population Covered","% Patients Covered (SEER Incidence)","% Urban Pop Covered",
    "% Rural Pop Covered","% Lowest Income Quartile Covered","% Highest Income Quartile Covered",
    "% Hispanic Pop Covered","% Black Pop Covered",
]
for c in pct_cols:
    if c in fmt.columns:
        fmt[c] = fmt[c].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "NA")

if "Covered Population (M)" in fmt.columns:
    fmt["Covered Population (M)"] = fmt["Covered Population (M)"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "NA")

# Save and display
out_path = f"out/asco_table_{RADIUS_MI}mi.csv"
fmt.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
fmt


Saved: out/asco_table_30mi.csv


,Cancer Type,No. Trials (NCT IDs),No. Sites (ZIPs),% Population Covered,Covered Population (M),% Patients Covered (SEER Incidence),% Urban Pop Covered,% Rural Pop Covered,% Lowest Income Quartile Covered,% Highest Income Quartile Covered,% Hispanic Pop Covered,% Black Pop Covered
0,any_gi,189,169,56.7%,189.57,NA,66.9%,5.9%,36.8%,82.3%,62.6%,68.0%
1,car_t,189,169,56.7%,189.57,NA,66.9%,5.9%,36.8%,82.3%,62.6%,68.0%


In [53]:
# --- NEW CELL AT THE BOTTOM ---
def compare_rarity_groups(radius=30):
    results = []
    # Identify which columns actually exist in zip_universe to avoid KeyErrors
    existing_common = [f"covered_{k}_{radius}mi" for k in COMMON_CANCERS if f"covered_{k}_{radius}mi" in zip_universe.columns]
    existing_rare = [f"covered_{k}_{radius}mi" for k in RARE_CANCERS if f"covered_{k}_{radius}mi" in zip_universe.columns]

    group_logic = {
        "Total GI (Any)": zip_universe[f"covered_any_gi_{radius}mi"],
        "Common Cancers": zip_universe[existing_common].any(axis=1) if existing_common else pd.Series([False]*len(zip_universe)),
        "Rare Cancers": zip_universe[existing_rare].any(axis=1) if existing_rare else pd.Series([False]*len(zip_universe))
    }
    
    for label, mask in group_logic.items():
        zip_universe["temp_mask"] = mask
        results.append({
            "Group": label,
            "% Pop Covered": 100 * pop_share(zip_universe, "temp_mask"),
            "% Urban Covered": 100 * pop_share(zip_universe[zip_universe["rural_urban"]=="Urban"], "temp_mask"),
            "% Rural Covered": 100 * pop_share(zip_universe[zip_universe["rural_urban"]=="Rural"], "temp_mask"),
            "% Q1 Income Covered": 100 * pop_share(zip_universe[zip_universe["inc_quartile"]=="Q1 (Lowest)"], "temp_mask"),
            "% Q4 Income Covered": 100 * pop_share(zip_universe[zip_universe["inc_quartile"]=="Q4 (Highest)"], "temp_mask"),
            "% Black Covered": 100 * subgroup_coverage(zip_universe, "temp_mask", "pop_black"),
            "% Hispanic Covered": 100 * subgroup_coverage(zip_universe, "temp_mask", "pop_hispanic")
        })
    return pd.DataFrame(results)

rarity_comparison = compare_rarity_groups(30)
display(rarity_comparison)
rarity_comparison.to_csv("out/common_vs_rare_comparison_60mi.csv", index=False)

,Group,% Pop Covered,% Urban Covered,% Rural Covered,% Q1 Income Covered,% Q4 Income Covered,% Black Covered,% Hispanic Covered
0,Total GI (Any),83.614071,90.878452,47.453571,65.812335,97.956795,85.541448,84.216422
1,Common Cancers,83.410775,90.812742,46.565404,65.530950,97.939138,85.508476,84.149931
2,Rare Cancers,63.774005,71.537839,25.130515,44.791815,85.546810,68.467953,65.394939


In [50]:
import re

def map_funder_type(funder_str):
    if not isinstance(funder_str, str): return "Other"
    f = funder_str.upper().strip()
    
    # 1. NIH
    if re.search(r'\bNIH\b|\bNATIONAL INSTITUTES OF HEALTH\b', f):
        return "NIH"
    
    # 2. Industry
    if re.search(r'\bINDUSTRY\b', f):
        return "Industry"
    
    # 3. Fed (Federal Agencies)
    # \b ensures we match 'VA' as a word, not inside 'Advanced'
    if re.search(r'\bFED\b|\bU\.S\. FED\b|\bFDA\b|\bVA\b|\bDOD\b|\bFEDERAL\b', f):
        return "Fed"
    
    # 4. Other (Universities, Hospitals, Non-profits)
    return "Other"

In [52]:
# --- NEW CELL AT THE BOTTOM ---

# 1. Aggregate by Rarity (Common vs Rare vs Any GI)
def get_phase_enrollment_table(df, group_col):
    phase_order = ["Phase 1", "Phase 2", "Phase 3"]
    
    # Count Trials
    counts = (df.dropna(subset=["phase_norm"])
              .groupby([group_col, "phase_norm"])["nct_id"].nunique()
              .unstack(fill_value=0)
              .reindex(columns=phase_order, fill_value=0))
    counts.columns = [f"Trials {ph}" for ph in counts.columns]
    
    # Sum Enrollment
    enr = (df.groupby([group_col, "phase_norm"])["enrollment_num"]
           .sum(min_count=1)
           .unstack()
           .reindex(columns=phase_order))
    enr.columns = [f"Enrollment {ph}" for ph in enr.columns]
    
    return pd.concat([counts, enr], axis=1).reset_index()

# Prepare data for "Any GI" baseline (Unique NCT IDs)
gi_unique = (enroll_df.assign(phase_int=lambda d: d["phase_norm"].map({"Phase 1":1,"Phase 2":2,"Phase 3":3,"Phase 4":4}))
             .groupby("nct_id", as_index=False)
             .agg(phase_norm=("phase_norm","max"), enrollment_num=("enrollment_num","max"), rarity=("rarity","first"), funder_type=("funder_type","first")))

# Table A: Common vs Rare
rarity_table = get_phase_enrollment_table(enroll_df, "rarity")
# Add the 'Any GI' unique baseline row
any_gi_row = get_phase_enrollment_table(gi_unique.assign(rarity="Any GI (Unique)"), "rarity")
final_rarity_table = pd.concat([any_gi_row, rarity_table], ignore_index=True)

print(gi_unique.columns)
# Table B: By Funder Type (on unique trials)
funder_table = get_phase_enrollment_table(gi_unique, "funder_type")

# Save and Display
final_rarity_table.to_csv("out/phase_enrollment_common_vs_rare.csv", index=False)
funder_table.to_csv("out/phase_enrollment_by_funder.csv", index=False)

print("--- Common vs Rare Summary ---")
display(final_rarity_table)
print("\n--- Funder Type Summary ---")
display(funder_table)

Index(['nct_id', 'phase_norm', 'enrollment_num', 'rarity', 'funder_type'], dtype='object')
--- Common vs Rare Summary ---


,rarity,Trials Phase 1,Trials Phase 2,Trials Phase 3,Enrollment Phase 1,Enrollment Phase 2,Enrollment Phase 3
0,Any GI (Unique),66,174,43,2590,12671,23205
1,Common,60,150,38,2438,11344,20389
2,Rare,6,25,5,152,1424,2816



--- Funder Type Summary ---


,funder_type,Trials Phase 1,Trials Phase 2,Trials Phase 3,Enrollment Phase 1,Enrollment Phase 2,Enrollment Phase 3
0,Fed,0,2,0,NaN,156.0,NaN
1,Industry,12,44,30,1023.0,6315.0,17789.0
2,NIH,6,9,1,185.0,633.0,224.0
3,Other,48,119,12,1382.0,5567.0,5192.0


Combining gastric , gastroesophagal and esophageal as one gastroesophagal

In [49]:
# ==============================
# Page-2 style coverage table (MERGED GASTROESOPHAGEAL VERSION)
# ==============================
import numpy as np
import pandas as pd

# ---- Params
RADIUS_MI = 30   # <- set 30 or 60
INCLUDE_OTHER_GI = False

# ---- Merge Definitions
UPPER_GI_KEYS = ["gastric_cancer", "gastroesophageal_cancer", "esophageal_cancer"]
MERGED_TARGET = "gastroesophageal_cancer"

# ---- Label map
CANCER_LABELS = LABELS.copy()
CANCER_LABELS[MERGED_TARGET] = "Gastroesophageal Cancer"
if INCLUDE_OTHER_GI:
    CANCER_LABELS["other_gi"] = "Other GI"

# ---- Safety checks
for need in ["master","trial_sites","zip_universe"]:
    if need not in globals():
        raise RuntimeError(f"Missing `{need}`. Run the combine/ZIP-universe steps first.")

# Ensure numerics
zip_universe["population"]   = pd.to_numeric(zip_universe["population"], errors="coerce")
zip_universe["pop_hispanic"] = pd.to_numeric(zip_universe.get("pop_hispanic", np.nan), errors="coerce")
zip_universe["pop_black"]    = pd.to_numeric(zip_universe.get("pop_black", np.nan), errors="coerce")

# ---- 1. Create Merged Coverage Flag
# This ensures a ZIP is 'covered' if it is within range of ANY of the three merged types
col_target = f"covered_{MERGED_TARGET}_{RADIUS_MI}mi"
cols_to_union = [f"covered_{k}_{RADIUS_MI}mi" for k in UPPER_GI_KEYS if f"covered_{k}_{RADIUS_MI}mi" in zip_universe.columns]
if cols_to_union:
    zip_universe[col_target] = zip_universe[cols_to_union].any(axis=1)

# ---- 2. Create Merged Dataframes for Count Logic
# This prevents double-counting trials (NCT IDs) that were listed in multiple sub-categories
master_tbl = master.copy()
master_tbl.loc[master_tbl["cancer_type"].isin(UPPER_GI_KEYS), "cancer_type"] = MERGED_TARGET
master_tbl = master_tbl.drop_duplicates(["nct_id", "cancer_type"])

sites_tbl = trial_sites.copy()
sites_tbl.loc[sites_tbl["cancer_type"].isin(UPPER_GI_KEYS), "cancer_type"] = MERGED_TARGET
sites_tbl = sites_tbl.drop_duplicates(["zip", "cancer_type"])

# ---- Helpers
def pop_share(df: pd.DataFrame, covered_col: str, weight_col: str = "population") -> float:
    if covered_col not in df.columns or weight_col not in df.columns or df.empty:
        return float("nan")
    w = pd.to_numeric(df[weight_col], errors="coerce").fillna(0.0)
    den = w.sum()
    if den <= 0: return float("nan")
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / den)

def covered_population_millions(df: pd.DataFrame, covered_col: str) -> float:
    if covered_col not in df.columns or "population" not in df.columns:
        return float("nan")
    w = pd.to_numeric(df["population"], errors="coerce").fillna(0.0)
    c = df[covered_col].astype(bool)
    return float((w[c]).sum() / 1_000_000.0)

def counts_for(key: str) -> tuple[int, int]:
    if key == "any_gi":
        return int(master["nct_id"].nunique()), int(trial_sites["zip"].nunique())
    tf = master_tbl[master_tbl["cancer_type"] == key]
    ts = sites_tbl[sites_tbl["cancer_type"] == key]
    return int(tf["nct_id"].nunique()), int(ts["zip"].nunique())

# ---- Build rows
rows = []

# Clean up priority lists to prevent duplicates in the table
COMMON_CLEAN = (COMMON_CANCERS - set(UPPER_GI_KEYS)) | {MERGED_TARGET}
RARE_CLEAN = RARE_CANCERS - set(UPPER_GI_KEYS)

priority_order = ["any_gi"] + sorted(list(COMMON_CLEAN)) + sorted(list(RARE_CLEAN))
exists = set(master_tbl["cancer_type"].unique())
ordered_keys = [k for k in priority_order if k in exists or k == "any_gi"]

for key in ordered_keys:
    label = CANCER_LABELS.get(key, key)
    covered_col = f"covered_any_gi_{RADIUS_MI}mi" if key=="any_gi" else f"covered_{key}_{RADIUS_MI}mi"
    n_trials, n_sites = counts_for(key)

    pop_all  = pop_share(zip_universe, covered_col, "population")
    coveredM = covered_population_millions(zip_universe, covered_col)

    pop_urb = pop_share(zip_universe[zip_universe["rural_urban"]=="Urban"], covered_col) if "rural_urban" in zip_universe.columns else np.nan
    pop_rur = pop_share(zip_universe[zip_universe["rural_urban"]=="Rural"], covered_col) if "rural_urban" in zip_universe.columns else np.nan

    if "inc_quartile" in zip_universe.columns:
        pop_q1 = pop_share(zip_universe[zip_universe["inc_quartile"]=="Q1 (Lowest)"], covered_col)
        pop_q4 = pop_share(zip_universe[zip_universe["inc_quartile"]=="Q4 (Highest)"], covered_col)
    else:
        pop_q1 = pop_q4 = np.nan

    hisp = subgroup_coverage(zip_universe, covered_col, "pop_hispanic")
    blk  = subgroup_coverage(zip_universe, covered_col, "pop_black")

    rows.append({
        "Cancer Type": label,
        "No. Trials (NCT IDs)": n_trials,
        "No. Sites (ZIPs)": n_sites,
        "% Population Covered": 100*pop_all if pd.notna(pop_all) else np.nan,
        "Covered Population (M)": coveredM,
        "% Urban Pop Covered": 100*pop_urb if pd.notna(pop_urb) else np.nan,
        "% Rural Pop Covered": 100*pop_rur if pd.notna(pop_rur) else np.nan,
        "% Lowest Income Quartile Covered": 100*pop_q1 if pd.notna(pop_q1) else np.nan,
        "% Highest Income Quartile Covered": 100*pop_q4 if pd.notna(pop_q4) else np.nan,
        "% Hispanic Pop Covered": 100*hisp if pd.notna(hisp) else np.nan,
        "% Black Pop Covered": 100*blk if pd.notna(blk) else np.nan,
    })

table = pd.DataFrame(rows)
fmt = table.copy()
pct_cols = ["% Population Covered","% Urban Pop Covered","% Rural Pop Covered",
            "% Lowest Income Quartile Covered","% Highest Income Quartile Covered",
            "% Hispanic Pop Covered","% Black Pop Covered"]

for c in pct_cols:
    fmt[c] = fmt[c].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "NA")
if "Covered Population (M)" in fmt.columns:
    fmt["Covered Population (M)"] = fmt["Covered Population (M)"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "NA")

out_path = f"out/asco_merged_table_{RADIUS_MI}mi.csv"
fmt.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
fmt

Saved: out/asco_merged_table_30mi.csv


,Cancer Type,No. Trials (NCT IDs),No. Sites (ZIPs),% Population Covered,Covered Population (M),% Urban Pop Covered,% Rural Pop Covered,% Lowest Income Quartile Covered,% Highest Income Quartile Covered,% Hispanic Pop Covered,% Black Pop Covered
0,any_gi,283,1322,83.6%,279.76,90.9%,47.5%,65.8%,98.0%,84.2%,85.5%
1,Colorectal_cancer,82,1026,80.2%,268.26,88.1%,40.8%,61.5%,97.0%,81.3%,83.4%
2,Gastroesophageal Cancer,46,684,71.9%,240.66,80.2%,30.9%,50.2%,92.7%,73.0%,74.9%
3,Hepatocellular_cancer,45,210,54.7%,182.91,64.3%,6.8%,35.5%,81.9%,62.1%,65.0%
4,Pancreatic_cancer,75,730,75.3%,251.94,83.8%,32.9%,54.9%,94.5%,76.4%,79.7%
5,Adenocarcinoma,2,419,43.6%,145.82,47.8%,22.5%,32.8%,51.5%,48.8%,40.3%
6,Anal_cancer,5,19,19.2%,64.13,22.8%,0.9%,13.4%,32.0%,20.0%,31.0%
7,Appendiceal_cancer,1,1,0.5%,1.62,0.6%,0.1%,0.3%,0.7%,0.7%,0.6%
8,BiliaryDuct_cancer,28,109,44.8%,149.89,52.8%,4.9%,29.7%,67.0%,49.5%,55.3%


In [58]:
# --- INDIVIDUAL CANCERS WITH FUNDER TYPE AS COLUMNS ---

# 1. Prepare merged dataframe and deduplicate
# (Matches the gastroesophageal merge logic from previous steps)
UPPER_GI_KEYS = ["gastric_cancer", "esophageal_cancer", "gastroesophageal_cancer"]
MERGED_KEY = "gastroesophageal_cancer"

merged_funder_df = enroll_df.copy()
merged_funder_df.loc[merged_funder_df["cancer_type"].isin(UPPER_GI_KEYS), "cancer_type"] = MERGED_KEY
merged_funder_df = merged_funder_df.drop_duplicates(subset=["nct_id", "cancer_type"], keep="first")

# 2. Pivot the Trial Counts by Funder Type
funder_cols = ["Industry", "NIH", "Fed", "Other"]

funder_trials_pivot = (merged_funder_df
                       .groupby(["cancer_type", "funder_type"])["nct_id"].nunique()
                       .unstack(fill_value=0)
                       .reindex(columns=funder_cols, fill_value=0))

# Rename columns for clarity
funder_trials_pivot.columns = [f"{c} Trials" for c in funder_trials_pivot.columns]

# 3. Pivot the Enrollment Sums by Funder Type
funder_enroll_pivot = (merged_funder_df
                       .groupby(["cancer_type", "funder_type"])["enrollment_num"].sum()
                       .unstack(fill_value=0)
                       .reindex(columns=funder_cols, fill_value=0))

funder_enroll_pivot.columns = [f"{c} Enrollment" for c in funder_enroll_pivot.columns]

# 4. Combine with the Phase Summary
# First get the standard phase table
phase_table = get_phase_enrollment_table(merged_funder_df, "cancer_type").set_index("cancer_type")

# Merge everything together
individual_wide_table = pd.concat([phase_table, funder_trials_pivot, funder_enroll_pivot], axis=1).reset_index()

# 5. Add 'Any GI' Baseline Row
# Calculate baseline for funders using gi_unique (the unique trials across all cancers)
any_gi_funder_trials = gi_unique["funder_type"].value_counts().reindex(funder_cols, fill_value=0)
any_gi_funder_enroll = gi_unique.groupby("funder_type")["enrollment_num"].sum().reindex(funder_cols, fill_value=0)

any_gi_phase = get_phase_enrollment_table(gi_unique.assign(cancer_type="any_gi"), "cancer_type")

# Create the full baseline row
any_gi_row = any_gi_phase.copy()
for col in funder_cols:
    any_gi_row[f"{col} Trials"] = any_gi_funder_trials.get(col, 0)
    any_gi_row[f"{col} Enrollment"] = any_gi_funder_enroll.get(col, 0)

# 6. Final Labeling and Cleanup
current_labels = LABELS.copy()
current_labels[MERGED_KEY] = "Gastroesophageal Cancer"

final_pivoted_table = pd.concat([any_gi_row, individual_wide_table], ignore_index=True)
final_pivoted_table.insert(0, "Cancer Label", final_pivoted_table["cancer_type"].map(current_labels).fillna("Any GI (Unique)"))
final_pivoted_table = final_pivoted_table.drop(columns=["cancer_type"])

# Save and Display
final_pivoted_table.to_csv("out/phase_enrollment_funder_columns.csv", index=False)
print("--- Individual Cancers: Phase & Funder Columns Summary ---")
display(final_pivoted_table)

--- Individual Cancers: Phase & Funder Columns Summary ---


,Cancer Label,Trials Phase 1,Trials Phase 2,Trials Phase 3,Enrollment Phase 1,Enrollment Phase 2,Enrollment Phase 3,Industry Trials,Industry Enrollment,NIH Trials,NIH Enrollment,Fed Trials,Fed Enrollment,Other Trials,Other Enrollment
0,Any GI (Unique),66,174,43,2590.0,12671.0,23205.0,86,25127,16,1042,2,156,179,12141
1,Adenocarcinoma,0,2,0,NaN,130.0,NaN,0,0,0,0,0,0,2,130
2,Anal_cancer,2,3,0,69.0,143.0,NaN,0,0,1,40,0,0,4,172
3,Appendiceal_cancer,0,1,0,NaN,39.0,NaN,0,0,0,0,0,0,1,39
4,BiliaryDuct_cancer,4,19,5,83.0,1112.0,2816.0,10,3257,3,161,0,0,15,593
5,Colorectal_cancer,20,49,13,881.0,4230.0,8749.0,28,8630,5,189,0,0,49,5041
6,Gastroesophageal Cancer,9,26,11,238.0,2831.0,5368.0,16,6045,1,224,2,156,27,2012
7,Hepatocellular_cancer,8,32,5,400.0,1963.0,2789.0,17,3597,2,130,0,0,26,1425
8,Pancreatic_cancer,23,43,9,919.0,2320.0,3483.0,16,3695,4,298,0,0,55,2729
